In [4]:
from datetime import datetime
from pathlib import Path

import pandas as pd
from evidently import DataDefinition, Dataset, Report
from evidently.presets import DataSummaryPreset

In [ ]:
ROOT_DIR = Path.cwd().parent
RAW_REFERENCE_DATA_PATH = ROOT_DIR / "data/processed/X_train.csv"
RAW_CURRENT_DATA_PATH = ROOT_DIR / "data/user_data/user_data_<datetime>.csv"     # replace <datetime>
REPORT_DIR = ROOT_DIR / "reports"

In [ ]:
reference_data_df = pd.read_csv(RAW_REFERENCE_DATA_PATH, encoding="utf-8")

current_data_df = pd.read_csv(RAW_CURRENT_DATA_PATH, encoding="utf-8")
current_data_df.drop(columns=["id"], inplace=True)

In [ ]:
data_definition = DataDefinition(
    numerical_columns = [
        "age",
        "Medu",
        "Fedu",
        "traveltime",
        "studytime",
        "failures",
        "famrel",
        "freetime",
        "goout",
        "Dalc",
        "Walc",
        "health",
        "absences",
    ],
    categorical_columns = [
        "school",
        "sex",
        "address",
        "famsize",
        "Pstatus",
        "Mjob",
        "Fjob",
        "reason",
        "guardian",
        "schoolsup",
        "famsup",
        "paid",
        "activities",
        "nursery",
        "higher",
        "internet",
        "romantic",
    ]
)

current_data_ds = Dataset.from_pandas(current_data_df, data_definition)
reference_data_ds = Dataset.from_pandas(reference_data_df, data_definition)

report = Report([DataSummaryPreset()])
result = report.run(current_data_ds, reference_data_ds)

# https://github.com/evidentlyai/evidently/issues/1595
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
result.save_html(filename=str(REPORT_DIR / f"drift_{timestamp}.html"))